# 2.4 Basic Flight Control - Fly to Car

## LLM Function Wrapper
The above LLM-based drone control function calls are wrapped in airsim_agent.py.

You only need to use 1 function:

In [ ]:
def process(self, command, run_python_code=True):
    #step 1, ask the language model
    response = self.ask(command)

    #step 2, extract Python code
    python_code = self.extract_python_code(response)

    #step 3, execute Python code
    if run_python_code and python_code:
        exec(python_code)
    return python_code

Usage:

In [ ]:
import airsim_agent
my_agent = airsim_agent.AirSimAgent()
command = "take off"
python_code = my_agent.process(command) # Don't execute code by default
print("python_code:", python_code)

In [ ]:
# Agent initialization function:
# def __init__(self, system_prompts="system_prompts/airsim_basic_cn.txt", knowledge_prompt="prompts/airsim_basic_cn.txt", chat_history=[]):


In [ ]:
# Write knowledge base to aisim_lession23.txt
kg_promot_file = "prompts/aisim_lession23.txt"

kg_prompt = """
Here are some functions you can use to command the drone.

aw.takeoff() - Take off the drone.
aw.land() - Land the drone.

Reply in the following example format:
```python
i=1  # output python code here
```
This code assigns a value.

You do not need to worry about importing aw; it is already declared in the environment.
"""

pt_file = open(kg_promot_file, "w", encoding="utf-8")
pt_file.write(kg_prompt)
pt_file.close()

In [ ]:
import airsim_agent
my_agent = airsim_agent.AirSimAgent(knowledge_prompt="prompts/aisim_lession23.txt")
command = "take off"
python_code = my_agent.process(command, True) # Execute code
print("python_code:", python_code)

## Fly to the Car

Since the drone has already taken off, we now need to fly it to the target position.

We can get the target position through the following function:
client.simGetObjectPose(object_name)

Since the targets are objects inside Unreal Engine and not directly accessible by name, we need a dictionary to convert them:

``` python
objects_dict = {
    "turbine1": "BP_Wind_Turbines_C_1",
    "turbine2": "StaticMeshActor_2",
    "solarpanels": "StaticMeshActor_146",
    "crowd": "StaticMeshActor_6",
    "car": "StaticMeshActor_10",
    "tower1": "SM_Electric_trellis_179",
    "tower2": "SM_Electric_trellis_7",
    "tower3": "SM_Electric_trellis_8",
}

```

Since we cannot use variables in the prompt, we need to include the target names and their descriptions in the prompt.

In [ ]:
# Build new prompt, write knowledge base to aisim_lession24.txt
kg_promot_file = "prompts/aisim_lession24.txt"

kg_prompt = """
Here are some functions you can use to command the drone.

aw.takeoff() - Take off the drone.
aw.land() - Land the drone.
aw.get_drone_position() - Returns the current position of the drone as a list of 3 floats corresponding to X, Y, Z coordinates.
aw.fly_to([x, y, z]) - Flies the drone to the specified position, given as a list of three arguments corresponding to X, Y, Z coordinates.
aw.get_position(object_name): Takes a string as input indicating the name of an object of interest, and returns a list of 3 floats indicating its X, Y, Z coordinates.

The following objects are in the scene, and you should refer to them using these exact names:

turbine1, turbine2, solarpanels, car, crowd, tower1, tower2, tower3.

Note: when calling functions, use the English names as follows:
turbine1: wind turbine 1
turbine2: wind turbine 2
solarpanels: solar panels
car: car
crowd: crowd
tower1: tower 1
tower2: tower 2
tower3: tower 3

For example, to get the position of turbine1, you should write:
aw.get_position("turbine1")
NOT:
aw.get_position("wind turbine 1")

None of the objects except for the drone itself are movable. Remember that there are two turbines and three towers. When there are multiple objects of the same type, and if I do not explicitly specify which object I am referring to, you should always ask me for clarification. Never make assumptions.

Reply in the following example format:
```python
i=1  # output python code here
```
This code assigns a value.

You do not need to worry about importing aw; it is already declared in the environment.
"""

pt_file = open(kg_promot_file, "w", encoding="utf-8")
pt_file.write(kg_prompt)
pt_file.close()

In [ ]:
# Build new prompt with NED coordinate system info, write to aisim_lession24.txt
kg_promot_file = "prompts/aisim_lession24.txt"

kg_prompt = """
Here are some functions you can use to command the drone.

aw.takeoff() - Take off the drone.
aw.land() - Land the drone.
aw.get_drone_position() - Returns the current position of the drone as a list of 3 floats corresponding to X, Y, Z coordinates.
aw.fly_to([x, y, z]) - Flies the drone to the specified position, given as a list of three arguments corresponding to X, Y, Z coordinates.
aw.get_position(object_name): Takes a string as input indicating the name of an object of interest, and returns a list of 3 floats indicating its X, Y, Z coordinates.

The following objects are in the scene, and you should refer to them using these exact names:

turbine1, turbine2, solarpanels, car, crowd, tower1, tower2, tower3.

For example, to get the position of turbine1, you should write:
aw.get_position("turbine1")

In terms of axis conventions, we use the NED (North-East-Down) coordinate system, where +X is North, +Y is East, and +Z is Down. This means the higher you fly, the more negative the Z value becomes. If the origin is on the ground, Z is zero at ground level and negative above ground. To fly up, subtract a value from the Z axis.

None of the objects except for the drone itself are movable. Remember that there are two turbines and three towers. When there are multiple objects of the same type, and if I do not explicitly specify which object I am referring to, you should always ask me for clarification. Never make assumptions.

Reply in the following example format:
```python
i=1  # output python code here
```
This code assigns a value.

You do not need to worry about importing aw; it is already declared in the environment.
"""

pt_file = open(kg_promot_file, "w", encoding="utf-8")
pt_file.write(kg_prompt)
pt_file.close()

In [ ]:
import airsim_agent
my_agent = airsim_agent.AirSimAgent(knowledge_prompt="prompts/aisim_lession24.txt")
command = "take off"
python_code = my_agent.process(command, True) # Execute code
print("python_code: \n", python_code)

In [ ]:
command = "fly to above the car"
python_code = my_agent.process(command, False) # Debug mode, don't execute code
print("python_code: \n", python_code)

In [ ]:
command = "fly to above the car"
python_code = my_agent.process(command, True) # Execute code
print("python_code: \n", python_code)

In [ ]:
2025